In [1]:
from langchain_aws import ChatBedrockConverse
# LLM setup for personalized report generation
llm = ChatBedrockConverse(
    model_id="amazon.nova-lite-v1:0",  # AWS Bedrock model ID
    region_name="us-east-1",           # AWS region
    temperature=0.7,                   # More creative output
    max_tokens=100                     # Generate a message with sufficient detail
)

In [2]:
import yaml
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
# --- Define Persona State ---
class PersonaState(TypedDict):
   persona: dict
   query: str
   response: str
# --- Load Persona Blueprint ---
with open("persona_blueprint.yaml", "r") as file:
   persona_config = yaml.safe_load(file)
# --- Nodes ---
def introduce_persona(state: PersonaState):
   agent = state["persona"]["agent"]
   print(f" Hello! I am {agent['name']}, your {agent['designation']}.")
   print(f"My purpose: {state['persona']['purpose']['description']}")
   return state

In [3]:
def execute_usecase(state: PersonaState):
    usecase = state["persona"]["use_case"]
    query = state["query"]

    # Build a persona-aware prompt
    prompt = f"""
    You are {state['persona']['agent']['name']}, a {state['persona']['agent']['designation']}.
    Purpose: {state['persona']['purpose']['description']}
    Scope: {', '.join(state['persona']['use_case']['scope'])}
    
    User query: {query}
    Domain: {usecase['domain']}
    
    Please respond in your persona style.
    """

    # Call the LLM
    response = llm.invoke(prompt)
    state["response"] = response.content

    print("Response:", state["response"])
    return state


In [4]:

def infer_behavior(state: PersonaState):
    tone = state["persona"]["interaction"]["human_interaction"]["tone"]
    prompt = f"You are a {tone} advisor. User asked: {state['query']}. Respond briefly."
    state["response"] = llm.invoke([HumanMessage(content=prompt)]).content
    print(" Inferred Response:", state["response"])
    return state

In [5]:
# --- Build Graph ---
graph = StateGraph(PersonaState)
graph.add_node("introduce", introduce_persona)
graph.add_node("usecase", execute_usecase)
graph.add_edge(START, "introduce")
graph.add_edge("introduce", "usecase")
graph.add_edge("usecase", END)
app = graph.compile()

In [6]:

result = app.invoke({"persona": persona_config, "query": "tell me about best stocks in market"})


 Hello! I am FinBot, your Personal Finance Advisor.
My purpose: Assist customers with personalized investment and savings strategies.
Response: Hello there! I'm FinBot, your Personal Finance Advisor, and I'm here to help you navigate the exciting world of investments. When it comes to selecting the best stocks in the market, it's crucial to remember that what works best for one person may not necessarily work for another. However, I can certainly provide you with some insights into some top-performing sectors and stocks that have been making headlines recently.

**Current Market Highlights (As of the latest data):**

### 
